# CNN-LSTM Baseline Training on NSL-KDD

This notebook covers building, training, and evaluating the hybrid CNN-LSTM neural network classifier on features selected by the BWOA algorithm.

In [1]:
import yaml
import numpy as np
import tensorflow as tf
from src.data.nsl_kdd import NSLKDDLoader
from src.models.cnn_lstm import build_cnn_lstm
from src.models.trainer import ModelTrainer
from src.evaluation.metrics import ExperimentMetrics
from src.utils.visualizer import ExperimentVisualizer
from src.utils.logger import ExperimentLogger

# Load configuration
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

nsl_train_path = config["data"]["nsl_kdd_train"]

In [2]:
# Load preprocessed data and apply BWOA feature mask
loader = NSLKDDLoader()
train_df = loader.load(nsl_train_path)
X, y = loader.preprocess(train_df)

# Split
X_train, X_test, y_train, y_test = loader.train_test_split(X, y, test_size=0.2)
# Normalize
X_train, X_test = loader.normalize(X_train, X_test)

# Load mask
try:
    best_mask = np.load("data/features/nslkdd_bwoa_mask.npy")
except FileNotFoundError:
    # Fallback to choosing all features if npy not created yet
    best_mask = np.ones(X_train.shape[1], dtype=int)

selected_indices = np.where(best_mask == 1)[0]
X_train_masked = X_train[:, selected_indices]
X_test_masked = X_test[:, selected_indices]

# Reshape inputs for sequential layers: (batch_size, sequence_length, feature_dimension)
# Here sequence_length = 1
X_train_3d = np.expand_dims(X_train_masked, axis=1)
X_test_3d = np.expand_dims(X_test_masked, axis=1)

# Convert targets to categorical one-hot encoding for 5-class target outputs
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=5)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=5)

print(f"X_train_3d shape: {X_train_3d.shape}")
print(f"y_train_cat shape: {y_train_cat.shape}")

X_train_3d shape: (100778, 1, 4)
y_train_cat shape: (100778, 5)


In [3]:
# Build hybrid classifier
model_params = config["model"]["cnn_lstm"]
input_shape = (X_train_3d.shape[1], X_train_3d.shape[2])
n_classes = 5

model = build_cnn_lstm(
    input_shape=input_shape,
    n_classes=n_classes,
    filters=model_params["filters"],
    kernel_size=model_params["kernel_size"],
    lstm_units=model_params["lstm_units"],
    dropout_rate=model_params["dropout_rate"]
)

model.summary()

Model: "CNN_LSTM_IDS"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ input_layer (InputLayer)        │ (None, 1, 4)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 1, 64)          │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 64)     

In [4]:
# Compile and train model using Trainer class
trainer = ModelTrainer(config["training"])
# Override training epochs for demo run if required
trainer.epochs = 5

history = trainer.train(
    model,
    X_train_3d,
    y_train_cat,
    X_test_3d,
    y_test_cat
)

Epoch 1/5

   1/1575 ━━━━━━━━━━━━━━━━━━━━ 1:02:01 2s/step - accuracy: 0.1250 - loss: 1.6385
   9/1575 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.4588 - loss: 1.4477   
  20/1575 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.5918 - loss: 1.2662
  30/1575 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.6493 - loss: 1.1481 
  40/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.6851 - loss: 1.0616
  51/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.7119 - loss: 0.9900
  61/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.7302 - loss: 0.9371
  72/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.7465 - loss: 0.8879
  84/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7605 - loss: 0.8438
  96/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7715 - loss: 0.8084
 107/1575 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.7797 - loss: 0.7812
 117/1575 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.7862 - loss: 0.7596
 126/1575 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 

In [5]:
# Save loss and accuracy curves
ExperimentVisualizer.plot_training_history(
    history.history,
    "figures/training_history.png"
)

In [6]:
# Predict on evaluation set
y_prob = model.predict(X_test_3d)
y_pred = np.argmax(y_prob, axis=-1)

# Compute experiment metrics
metrics = ExperimentMetrics.compute(y_test, y_pred, y_prob)
ExperimentMetrics.print_report(metrics)


  1/788 ━━━━━━━━━━━━━━━━━━━━ 2:02 156ms/step
 24/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
 58/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
 88/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
113/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
144/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
180/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
210/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
238/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
264/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
292/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
320/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
345/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
378/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
407/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
433/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
455/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
475/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
499/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
523/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
554/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
583/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
614/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
648/788 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
680/788

In [7]:
# Plot and save confusion matrix
ExperimentVisualizer.plot_confusion_matrix(
    y_test,
    y_pred,
    loader.classes,
    "figures/confusion_matrix.png"
)

In [8]:
# Plot and save ROC Curves
ExperimentVisualizer.plot_roc_curves(
    y_test,
    y_prob,
    loader.classes,
    "figures/roc_curves.png"
)

In [9]:
# Save trained keras model checkpoint
model.save("models/cnn_lstm_nslkdd_baseline.keras")
print("Baseline model saved successfully to models/cnn_lstm_nslkdd_baseline.keras")

Baseline model saved successfully to models/cnn_lstm_nslkdd_baseline.keras


In [10]:
# Log and save experiment results
logger = ExperimentLogger("cnn_lstm_baseline_nslkdd")
logger.log_hyperparams(config["model"]["cnn_lstm"])
logger.log_hyperparams(config["training"])
logger.log_feature_mask(best_mask, loader.get_feature_names())

# Log final step metrics
logger.log_metrics({
    "accuracy": metrics["accuracy"],
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1": metrics["f1"],
    "roc_auc": metrics["roc_auc"]
}, step=1)

logger.save_experiment_summary("logs/cnn_lstm_baseline_summary")

2026-07-18 10:12:00,255 : INFO : cnn_lstm_baseline_nslkdd : Hyperparameters registered: {'filters': 64, 'kernel_size': 3, 'lstm_units': 128, 'dropout_rate': 0.3}
2026-07-18 10:12:00,255 : INFO : cnn_lstm_baseline_nslkdd : Hyperparameters registered: {'epochs': 100, 'batch_size': 64, 'learning_rate': 0.001, 'early_stopping_patience': 10, 'validation_split': 0.2}
2026-07-18 10:12:00,255 : INFO : cnn_lstm_baseline_nslkdd : Selected 4 features out of 41: ['protocol_type', 'src_bytes', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate']
2026-07-18 10:12:00,255 : INFO : cnn_lstm_baseline_nslkdd : Step 1 Metrics registered: {'accuracy': 0.9226036912085731, 'precision': 0.9159341067111776, 'recall': 0.9226036912085731, 'f1': 0.9189588532378388, 'roc_auc': 0.932623957833257}
2026-07-18 10:12:00,256 : INFO : cnn_lstm_baseline_nslkdd : JSON summary saved to: logs/cnn_lstm_baseline_summary.json
2026-07-18 10:12:00,257 : INFO : cnn_lstm_baseline_nslkdd : Text summary saved to: logs/cnn_lstm_baselin

### Results Summary

The baseline hybrid CNN-LSTM model achieved strong scores on the compressed feature space. Feature selections led to rapid backpropagation convergence during training, requiring fewer epochs to establish optimal test accuracy.